    Instructions:
    1. Run task_0a.py to generate the vector database if not already generated.
    2. Press Run All (or restart kernel and run all cells).
    3. You will be prompted to provide input values.


    – Task 3 (LS1): Implement a program which (a) given one of the feature models, 
    (b) a user specified value of k, (c) one of the four dimensionality reduction 
    techniques (SVD, NNMF, LDA, k-means) chosen by the user, reports the top-k 
    latent semantics extracted under the selected feature space.

    – Store the latent semantics in a properly named output file

    – List imageID-weight pairs, ordered in decreasing order of weights

In [16]:
FEATURE_SPACE = input("Provide a feature space [color, hog, avgpool, layer3, fc, resnet_output].")

DIM_REDUCTION = input("Provide a dimensionality reduction technique [svd, nnmf, lda, kmeans].")

K = int(input("Enter K, the top K latent semantics to extract for the selected feature space."))

In [17]:
from utils.database_utils import retrieve
feature_vectors = retrieve(f'{FEATURE_SPACE}.pt')

print("Generating top-", K, " latent semantics under ", FEATURE_SPACE, " feature space using: ", DIM_REDUCTION)

Generating top- 5  latent semantics under  resnet_output  feature space using:  kmeans


In [18]:
if DIM_REDUCTION == "svd":
    from feature_reducers.svd import SVDReducer
    reducer = SVDReducer

elif DIM_REDUCTION == "nnmf":
    from feature_reducers.nnmf import NNMFReducer
    reducer = NNMFReducer

elif DIM_REDUCTION == "lda":
    from feature_reducers.lda import LDAReducer
    reducer = LDAReducer

else:
    # kmeans.
    from feature_reducers.kmeans import KMeansReducer
    reducer = KMeansReducer

reducer = reducer(feature_vectors, K)

similarity_matrix = reducer.get_similarity_matrix(feature_vectors)

latent_semantics = reducer.reduce_features(feature_vectors)
print("Top K latent semantics: ")
print(latent_semantics)
print("Shape: ", latent_semantics.shape)

Top K latent semantics: 
[[46.05407333 41.68006134 32.40818787 40.06840134 42.0682373 ]
 [59.28167343 57.25667953 48.38655472 60.13710022 58.60972595]
 [46.49497223 42.61494827 27.23882675 40.57872391 43.65811157]
 ...
 [43.44541931 41.68369675 46.30450439 37.53442001 43.20550919]
 [40.59019852 37.95668793 45.68091965 31.08154297 38.9336319 ]
 [42.95098495 38.68756104 46.52000046 37.47573853 40.6389389 ]]
Shape:  (4339, 5)


In [19]:
# Store the latent semantics in a properly named file.
# We opt to store just the reducer, as we anyway can generate the latent space quickly
# by loading the feature space and passing it to the reducer, eg:
#
# unpicked_reducer = retrieve(f'LS1_color_svd_reducer.pt')
# feature_vectors = retrieve(f'color.pt')
#
# unpickled_reducer.reduce_features(feature_vectors)

from utils.database_utils import store

store(reducer, f'LS1_{FEATURE_SPACE}_{K}_{DIM_REDUCTION}_reducer.pt')


 Saving:  LS1_resnet_output_5_kmeans_reducer.pt 



In [20]:
# List imageID-weight pairs, ordered in decreasing order of weights

# We are to showcase which images contribute more to each latent feature.
# This is taking the object-feature factor matrix, and sorting by each
# latent feature's weight.

image_weight_tuples = list(zip(feature_vectors.keys(), similarity_matrix))

print("Image ID - weight pairs sorted in descending order of weights for each latent feature:")

for i in range(K):
    print("\n\nLatent feature: ", i + 1)
    for IMG_ID, weight in sorted(image_weight_tuples, key=lambda x : x[1][i], reverse=True):
        print("(ID: ", IMG_ID, ", Weight: ", weight[i], end="),\t")

Image ID - weight pairs sorted in descending order of weights for each latent feature:


Latent feature:  1
(ID:  748 , Weight:  72.44503021240234),	(ID:  740 , Weight:  68.55806732177734),	(ID:  2952 , Weight:  67.23255920410156),	(ID:  3236 , Weight:  67.1196060180664),	(ID:  5812 , Weight:  65.6270751953125),	(ID:  2920 , Weight:  65.54706573486328),	(ID:  510 , Weight:  64.20902252197266),	(ID:  6254 , Weight:  63.690574645996094),	(ID:  6252 , Weight:  63.45071792602539),	(ID:  2870 , Weight:  62.325538635253906),	(ID:  5616 , Weight:  61.93153762817383),	(ID:  3952 , Weight:  61.6632080078125),	(ID:  722 , Weight:  61.41736602783203),	(ID:  300 , Weight:  61.41012954711914),	(ID:  7454 , Weight:  60.94120788574219),	(ID:  286 , Weight:  60.70695495605469),	(ID:  7744 , Weight:  60.24296951293945),	(ID:  320 , Weight:  59.67551803588867),	(ID:  3218 , Weight:  59.67313003540039),	(ID:  742 , Weight:  59.65201187133789),	(ID:  262 , Weight:  59.534420013427734),	(ID:  2 , Weight:  